In [26]:
!pip install faker


In [27]:
!pip install pandas faker requests

**The below code generates output without any cssv generation**

In [28]:
# =========================
# INSTALL DEPENDENCIES
# =========================
!pip install groq faker pandas

# =========================
# FULL PROGRAM (COLAB)
# =========================

import os
import pandas as pd
from faker import Faker
from groq import Groq
import random
import json

# -------------------------
# SET YOUR API KEY SECURELY
# -------------------------
# In Colab: set it like this instead of hardcoding:
# os.environ["GROQ_API_KEY"] = "YOUR_KEY_HERE"

api_key = "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"

client = Groq(api_key=api_key)
fake = Faker()



# -------------------------
# EXISTING TABLE (example)
# -------------------------
existing_df = pd.DataFrame({
    "student_id": [1, 2, 3],
    "name": ["Aarav", "Diya", "Kiran"],
    "age": [18, 19, 20],
    "grade": ["A", "B", "A"]
})

# -------------------------
# CONSTRAINTS
# -------------------------
CONSTRAINTS = {
    "age_min": 17,
    "age_max": 22,
    "grades": ["A", "B", "C", "A+", "B+"],
    "num_rows_to_generate": 20
}

# -------------------------
# LLM-BASED GENERATION
# -------------------------
def generate_with_llm(seed_table, constraints, n=10):
    prompt = f"""
You are generating synthetic student data.

Existing table:
{seed_table.to_json(orient='records')}

Constraints:
- Age must be between {constraints['age_min']} and {constraints['age_max']}
- Grades allowed: {constraints['grades']}
- Generate {n} new students
- Output ONLY valid JSON list of objects
- Each object must have: student_id, name, age, grade
- Ensure realistic Indian-style names

Return only JSON.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You generate structured synthetic datasets."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )

    content = response.choices[0].message.content
    return json.loads(content)


# -------------------------
# FALLBACK FAKE GENERATOR (Faker)
# -------------------------
def generate_with_faker(start_id, constraints, n=10):
    data = []
    for i in range(n):
        data.append({
            "student_id": start_id + i,
            "name": fake.name(),
            "age": random.randint(constraints["age_min"], constraints["age_max"]),
            "grade": random.choice(constraints["grades"])
        })
    return data


# -------------------------
# MAIN PIPELINE
# -------------------------
def generate_student_dataset(existing_df, constraints):
    try:
        print("Trying LLM generation via Groq...")
        new_data = generate_with_llm(existing_df, constraints, constraints["num_rows_to_generate"])
    except Exception as e:
        print("LLM failed, using Faker fallback:", e)
        start_id = int(existing_df["student_id"].max()) + 1
        new_data = generate_with_faker(start_id, constraints, constraints["num_rows_to_generate"])

    new_df = pd.DataFrame(new_data)

    combined = pd.concat([existing_df, new_df], ignore_index=True)
    return combined


# -------------------------
# RUN
# -------------------------
final_df = generate_student_dataset(existing_df, CONSTRAINTS)

print(final_df)

Trying LLM generation via Groq...
    student_id       name  age grade
0            1      Aarav   18     A
1            2       Diya   19     B
2            3      Kiran   20     A
3            4      Rohan   18     B
4            5     Saanvi   19     A
5            6     Vihaan   20     C
6            7      Aryan   21    B+
7            8       Myra   17     A
8            9      Kabir   18     B
9           10      Kiara   19    A+
10          11   Shivansh   20     C
11          12     Ananya   21     B
12          13    Reyansh   17     A
13          14       Zara   18    B+
14          15  Siddharth   19     A
15          16     Nalini   20     C
16          17     Vedant   21     B
17          18     Yashvi   17    A+
18          19    Dhairya   18     B
19          20     Suhana   19     C
20          21     Raghav   20     A
21          22      Aisha   21    B+
22          23    Shaurya   17     A


# The below code generates 2 files.

Input file- students.csv &
output file- fakerdata.csv text

In [29]:
# =========================
# INSTALL DEPENDENCIES
# =========================
!pip install groq faker pandas

import pandas as pd
import random
import json
from faker import Faker
from groq import Groq

fake = Faker()

# =========================
# API KEY (PUT YOUR KEY HERE)
# =========================
api_key = "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"
client = Groq(api_key=api_key)

# =========================
# STEP 1: CREATE INPUT FILE (students.csv)
# =========================
input_data = pd.DataFrame({
    "student_id": [1, 2, 3, 4, 5],
    "name": ["Aarav", "Diya", "Kiran", "Meera", "Rohan"],
    "age": [18, 19, 20, 21, 18],
    "grade": ["A", "B", "A", "C", "B"]
})

input_file = "students.csv"
input_data.to_csv(input_file, index=False)

print("Created input file:", input_file)

# =========================
# STEP 2: LOAD INPUT FILE
# =========================
existing_df = pd.read_csv(input_file)

# =========================
# CONSTRAINTS
# =========================
CONSTRAINTS = {
    "age_min": 17,
    "age_max": 22,
    "grades": ["A", "B", "C", "A+", "B+"],
    "num_rows_to_generate": 20
}

# =========================
# STEP 3: LLM GENERATION
# =========================
def generate_with_llm(seed_table, constraints, n):
    prompt = f"""
Generate synthetic student data.

Rules:
- Age between {constraints['age_min']} and {constraints['age_max']}
- Grades: {constraints['grades']}
- Generate {n} students
- Output ONLY valid JSON array
- Fields: student_id, name, age, grade

Existing data:
{seed_table.to_dict(orient='records')}
"""

    resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Return ONLY JSON array."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.5
    )

    text = resp.choices[0].message.content

    start = text.find("[")
    end = text.rfind("]") + 1
    return json.loads(text[start:end])

# =========================
# STEP 4: FALLBACK (FAKER)
# =========================
def generate_with_faker(start_id, constraints, n):
    return [
        {
            "student_id": start_id + i,
            "name": fake.name(),
            "age": random.randint(constraints["age_min"], constraints["age_max"]),
            "grade": random.choice(constraints["grades"])
        }
        for i in range(n)
    ]

# =========================
# STEP 5: PIPELINE
# =========================
def generate_dataset():
    try:
        print("Generating using Groq LLM...")
        new_data = generate_with_llm(
            existing_df,
            CONSTRAINTS,
            CONSTRAINTS["num_rows_to_generate"]
        )
    except Exception as e:
        print("LLM failed → using Faker:", e)
        start_id = existing_df["student_id"].max() + 1
        new_data = generate_with_faker(
            start_id,
            CONSTRAINTS,
            CONSTRAINTS["num_rows_to_generate"]
        )

    return pd.DataFrame(new_data)

# =========================
# STEP 6: RUN GENERATION
# =========================
output_df = generate_dataset()

# =========================
# STEP 7: SAVE OUTPUT FILE
# =========================
output_file = "fakerdata.csv"
output_df.to_csv(output_file, index=False)

print("Output file created:", output_file)
print(output_df)

Created input file: students.csv
Generating using Groq LLM...
Output file created: fakerdata.csv
    student_id       name  age grade
0            6      Ethan   19    A+
1            7       Liam   20    B+
2            8       Noah   17     A
3            9     Oliver   21     B
4           10        Ava   18     C
5           11   Isabella   19    A+
6           12        Mia   20     B
7           13  Charlotte   22     A
8           14    Abigail   18    B+
9           15      Emily   19     C
10          16     Harper   20     A
11          17     Evelyn   21     B
12          18       Lily   18    A+
13          19    Madison   19    B+
14          20   Victoria   20     C
15          21    Jessica   21     A
16          22   Samantha   18     B
17          23      Avery   19    A+
18          24      Riley   20    B+
19          25       Zoey   21     C
